# Infosys Quarterly Report Analysis

#### Developed By: Manaranjan Pradhan
#### www.manaranjanp.com

*This Jupyter notebook is confidential and proprietary to Manaranjan Pradhan. It is intended solely for authorized training purposes. Unauthorized distribution, sharing, or reproduction of this notebook or its contents is strictly prohibited. This material is for personal learning within the training program only and may not be used for commercial purposes or shared with others. Unauthorized use may result in disciplinary action or legal consequences. If you have received this notebook without authorization, please contact manaranjan@gmail.com immediately and delete all copies.*

In [35]:
pip install --quiet getpass4==0.0.14.1 pypdf==6.7.1 openai==2.21.0 faiss-cpu==1.13.2 llama-index==0.14.14 llama-index-readers-file==0.5.6 llama-index-vector-stores-faiss==0.5.3 llama-index-llms-groq==0.4.1 llama-index-embeddings-huggingface==0.6.1 transformers==4.51.3

### It will ask to restart the session. So, restart the session before proceeding to next step.

In [36]:
import nest_asyncio

nest_asyncio.apply()

In [37]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [38]:
from llama_index.core import SimpleDirectoryReader, ServiceContext, VectorStoreIndex, StorageContext
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.node_parser import (SentenceWindowNodeParser,)
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.core import Document
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [39]:
import faiss

## Configure LLM service

In [40]:
#llm = OpenAI(temperature=0,
#             model="chatgpt-4o-latest",
#             max_tokens=500)


llm = Groq(model="llama-3.3-70b-versatile")

In [41]:
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [42]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)
Settings.num_output = 512
Settings.context_window = 3900

## Load data
Downloaded from

https://www.infosys.com/investors/reports-filings/quarterly-results.html

In [ ]:
!pip install --quiet docling

In [ ]:
from docling.document_converter import DocumentConverter

In [ ]:
converter = DocumentConverter()
result = converter.convert("/content/ifrs-inr-press-release.pdf")
result.document.export_to_markdown()


In [ ]:
import os

# Define the output directory
output_dir = "output_documents"
os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists

# Define output file path
output_path = os.path.join(output_dir, "converted_document.md")

# Save as Markdown
with open(output_path, "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

print(f"Document saved at: {output_path}")

In [43]:
#q1_2024 = SimpleDirectoryReader(
#    input_files=["/content/ifrs-inr-press-release.pdf"]
#).load_data()

q1_2024 = SimpleDirectoryReader(
    input_files=["/content/output_documents/converted_document.md"]
).load_data()

# Chunk, Vectorize and Index in FAISS

In [44]:
# dimensions of text-ada-embedding-002
d = 384
faiss_index = faiss.IndexFlatL2(d)

In [45]:
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
q1_2024_index = VectorStoreIndex.from_documents(q1_2024, storage_context=storage_context)

## Build query engines

In [46]:
q1_2024__engine = q1_2024_index.as_query_engine(similarity_top_k=2)

In [47]:
# Assuming you have already built the index and query engine
retriever = q1_2024__engine.retriever
retrieved_nodes = retriever.retrieve("What is the revenue growth for the quarter?")

# Print the retrieved chunks
for i, node in enumerate(retrieved_nodes):
    print(f"\n--- Chunk {i+1} ---")
    print(node.get_text())


--- Chunk 1 ---
945                          | 5,360                          |
| Basic EPS ( ` )                       | 14.37                          | 12.78                          |
| Diluted EPS ( ` )                     | 14.35                          | 12.76                          |

<!-- image -->

## NOTES:

1. The above information is extracted from the audited condensed consolidated Balance sheet and Statement of Comprehensive Income for the quarter ended June 30, 2023, which have been taken on record at the Board meeting held on July 20, 2023.
2. A Fact Sheet providing the operating metrics of the Company can be downloaded from www.infosys.com.
3. Represents bank balance earmarked for final dividend. Payment date for dividend was July 3, 2023.
4. Other income is net of Finance Cost.

--- Chunk 2 ---
<!-- image -->

\

Solid Q1 year on year revenue growth of 4.2% at 20.8% operating margins Strong large deal closures and robust deal pipeline position us well for future 

## Run queries

In [48]:
response = q1_2024__engine.query(
    "What is the revenue growth for the quarter?"
)

In [49]:
q1_2024__engine.get_prompts()

{'response_synthesizer:text_qa_template': SelectorPromptTemplate(metadata={'prompt_type': <PromptType.QUESTION_ANSWER: 'text_qa'>}, template_vars=['context_str', 'query_str'], kwargs={}, output_parser=None, template_var_mappings={}, function_mappings={}, default_template=PromptTemplate(metadata={'prompt_type': <PromptType.QUESTION_ANSWER: 'text_qa'>}, template_vars=['context_str', 'query_str'], kwargs={}, output_parser=None, template_var_mappings=None, function_mappings=None, template='Context information is below.\n---------------------\n{context_str}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {query_str}\nAnswer: '), conditionals=[(<function is_chat_model at 0x7eb711112840>, ChatPromptTemplate(metadata={'prompt_type': <PromptType.CUSTOM: 'custom'>}, template_vars=['context_str', 'query_str'], kwargs={}, output_parser=None, template_var_mappings=None, function_mappings=None, message_templates=[ChatMessage(role=<MessageRole.

In [50]:
print(response)

The revenue growth for the quarter is 4.2% year on year and 1.0% sequentially in constant currency.


In [51]:
response = q1_2024__engine.query("Which are some of the key customer wins?")

In [52]:
print(response)

Some of the key customer wins include XacBank, for which Infosys Finacle helped transform their technology landscape with the Finacle Digital Banking Suite, enabling a robust digital foundation for the bank to achieve its growth strategy.


In [53]:
response = q1_2024__engine.query("What are some of the key achievements in the quarter?")

In [54]:
print(response)

Some of the key achievements in the quarter include revenues growing by 4.2% YoY and 1.0% QoQ in CC terms, reported revenues at ` 37,933 crore with a growth of 10.0% YoY, operating margin at 20.8% with a growth of 0.7% YoY, Basic EPS at ` 14.37 with a growth of 12.4% YoY, and FCF at ` 5,749 crore with a growth of 12.6% YoY and FCF conversion at 96.7% of net profit.


In [55]:
response = q1_2024__engine.query("What is the total assets?")

In [56]:
print(response)

131,322
